# Building Local OCR Application SmolDocling: A Step-by-Step Guide [Part 1]

Run cells in order with Python 3.11. Install dependencies in an isolated environment:

```bash
python -m pip install torch==2.15.0 transformers==5.22.1 docling-core==2.105.0 pillow==13.0.0 requests==2.34.2
```


In [ ]:
import os, json, hashlib, tempfile
from pathlib import Path
from importlib.metadata import version
import requests, torch
from PIL import Image
from docling_core.types.doc import DoclingDocument
from docling_core.types.doc.document import DocTagsDocument
from transformers import AutoProcessor, AutoModelForImageTextToText
MODEL_ID = 'docling-project/SmolDocling-256M-preview'
REVISION = 'ce51f56c4ebe36e0b1c3a55f67b261ba22a50bf8'
DEVICE = 'cpu'
DTYPE = torch.float32
torch.set_num_threads(4)
for package in ['torch', 'transformers', 'docling-core', 'pillow']:
    print(package, version(package))
print('Device:', DEVICE, 'dtype:', DTYPE)

In [ ]:
processor = AutoProcessor.from_pretrained(MODEL_ID, revision=REVISION, token=False)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, revision=REVISION, token=False, dtype=DTYPE,
    attn_implementation='eager').to(DEVICE).eval()
assert not model.training
print('Parameters:', sum(p.numel() for p in model.parameters()))
print('Checkpoint:', MODEL_ID, REVISION)

In [ ]:
# The same original presentation-slide example shown in the article.
IMAGE_URL = 'https://cdn-images-1.medium.com/max/800/1*Yep_LLJWj4rR8KeUoSCOIg.png'
image_dir = Path(tempfile.mkdtemp(prefix='tdb-smoldocling-'))
response = requests.get(IMAGE_URL, timeout=60)
response.raise_for_status()
image_path = image_dir / 'source-slide.png'
image_path.write_bytes(response.content)
image = Image.open(image_path).convert('RGB')
print('Image dimensions:', image.size)
print('Image SHA256:', hashlib.sha256(response.content).hexdigest())

In [ ]:
messages = [{'role': 'user', 'content': [
    {'type': 'image'}, {'type': 'text', 'text': 'Convert this page to docling.'}]}]
prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
inputs = processor(text=prompt, images=[image], return_tensors='pt').to(DEVICE)
print('Input tokens:', inputs.input_ids.shape[1])
print('Image tensors:', tuple(inputs.pixel_values.shape))

In [ ]:
with torch.inference_mode():
    generated_ids = model.generate(**inputs, max_new_tokens=8192, do_sample=False)
prompt_length = inputs.input_ids.shape[1]
trimmed_generated_ids = generated_ids[:, prompt_length:]
doctags = processor.batch_decode(trimmed_generated_ids, skip_special_tokens=False)[0].lstrip()
assert trimmed_generated_ids.shape[1] < 8192
assert '<doctag>' in doctags and '</doctag>' in doctags
print('Generated tokens:', trimmed_generated_ids.shape[1])
print(doctags)

In [ ]:
doctags_doc = DocTagsDocument.from_doctags_and_image_pairs([doctags], [image])
doc = DoclingDocument.load_from_doctags(doctags_doc, document_name='Document')
markdown = doc.export_to_markdown()
assert markdown.strip()
print(markdown)

In [ ]:
output_file = image_dir / 'output.html'
doc.save_as_html(output_file)
html = output_file.read_text()
assert '<html' in html.lower() and 'Document' in html
print('Saved output.html:', output_file.stat().st_size, 'bytes')

In [ ]:
def ocr_pipeline(input_image):
    if not isinstance(input_image, Image.Image) or input_image.width * input_image.height > 10_000_000:
        raise ValueError('Expected a PIL image with at most 10 million pixels')
    rgb = input_image.convert('RGB')
    batch = processor(text=prompt, images=[rgb], return_tensors='pt').to(DEVICE)
    with torch.inference_mode():
        generated = model.generate(**batch, max_new_tokens=8192, do_sample=False)
    output = generated[:, batch.input_ids.shape[1]:]
    if output.shape[1] >= 8192:
        raise RuntimeError('Generation reached the token limit')
    tags = processor.batch_decode(output, skip_special_tokens=False)[0].lstrip()
    parsed = DocTagsDocument.from_doctags_and_image_pairs([tags], [rgb])
    document = DoclingDocument.load_from_doctags(parsed, document_name='Document')
    return document.export_to_markdown()
wrapped = ocr_pipeline(image)
assert wrapped == markdown
print(wrapped)

In [ ]:
texts = [item.text for item, _ in doc.iterate_items() if hasattr(item, 'text')]
assert texts and all(isinstance(t, str) for t in texts)
expected_body = "Here I am writing down every word that I am going to say aloud when I give the presentation. All the text taking up space on the screen is going to make it really difficult for people to listen and to read everything at once, even though they are the same thing. It'd be better if I just went and added a couple key points on the slide and elaborate on them by referring to some flash cards or memorizing what I'm going to say."
assert ' '.join(markdown.split()) == expected_body
print('Slide body matches the source; the model omitted the heading.')
assert 'presentation' in html.lower()
assert torch.isfinite(model.get_input_embeddings().weight).all()
try: ocr_pipeline(None)
except ValueError: pass
else: raise AssertionError('Missing image accepted')
print('PASS: actual full-checkpoint CPU OCR, DocTags parsing, Markdown/HTML export and identical wrapper inference.')

## Original source-code and transcript appendix

Nonexecuting original material. Use the maintained cells above.

```text
!pip install torch -q
```

```text
import os
import torch
from docling_core.types.doc import DoclingDocument
from docling_core.types.doc.document import DocTagsDocument
from transformers import AutoProcessor, AutoModelForVision2Seq
from transformers.image_utils import load_image
```

```text
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
```

```text
processor = AutoProcessor.from_pretrained("ds4sd/SmolDocling-256M-preview")
model = AutoModelForVision2Seq.from_pretrained(
    "ds4sd/SmolDocling-256M-preview",
    torch_dtype=torch.bfloat16,
    #_attn_implementation="flash_attention_2" if DEVICE == "cuda" else "eager",
).to(DEVICE)
```

```text
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": "Convert this page to docling."}
        ]
    },
]
```

```text
prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
inputs = processor(text=prompt, images=[image], return_tensors="pt")
inputs = inputs.to(DEVICE)
```

```text
generated_ids = model.generate(**inputs, max_new_tokens=8192)
prompt_length = inputs.input_ids.shape[1]
trimmed_generated_ids = generated_ids[:, prompt_length:]
```

```text
doctags = processor.batch_decode(
    trimmed_generated_ids,
    skip_special_tokens=False,
)[0].lstrip()
```

```text
doctags_doc = DocTagsDocument.from_doctags_and_image_pairs([doctags], [image])
doc = DoclingDocument(name="Document")
doc.load_from_doctags(doctags_doc)
```

```text
from pathlib import Path
```

```text
def ocr_pipeline(image):
    # Load model and processor
    processor = AutoProcessor.from_pretrained("ds4sd/SmolDocling-256M-preview")
    model = AutoModelForVision2Seq.from_pretrained(
        "ds4sd/SmolDocling-256M-preview",
        torch_dtype=torch.bfloat16,
    ).to(DEVICE)

    # Prepare input message
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": "Convert this page to docling."}
            ]
        },
    ]
    
    # Process input
    prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(text=prompt, images=[image], return_tensors="pt").to(DEVICE)
    
    # Generate output
    generated_ids = model.generate(**inputs, max_new_tokens=8192)
    trimmed_generated_ids = generated_ids[:, inputs.input_ids.shape[1]:]
    
    # Decode output
    doctags = processor.batch_decode(trimmed_generated_ids, skip_special_tokens=False)[0].lstrip()
    
    # Create Docling document
    doctags_doc = DocTagsDocument.from_doctags_and_image_pairs([doctags], [image])
    doc = DoclingDocument(name="Document")
    doc.load_from_doctags(doctags_doc)
    
    return doc.export_to_markdown()
```

```text
You can find the codes and data used in this article in this GitHub Repo.
```

```text
<doctag><section_header_level_1><loc_167><loc_45><loc_333><loc_83>Don’t Do This!</section_header_level_1>
<text><loc_26><loc_145><loc_473><loc_399>Here I am writing down every word that I am going to say aloud when I give the presentation. All the text taking up space on the screen is going to make it really difficult for people to listen and to read everything at once, even though they are the same thing. It’d be better if I just went and added a couple key points on the slide and elaborate on them by referring to some flash cards or memorizing what I’m going to say.</text>
</doctag><end_of_utterance>
## Don’t Do This!
```

```text
Here I am writing down every word that I am going to say aloud when I give the presentation. All the text taking up space on the screen is going to make it really difficult for people to listen and to read everything at once, even though they are the same thing. It’d be better if I just went and added a couple key points on the slide and elaborate on them by referring to some flash cards or memorizing what I’m going to say.
```